<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ crime buffer scoring ★ </h3>
  <p> This notebook is used to consolidate mappings back into edges and give each edge a score. </p>
  <h4> Note: do not run on filtered edges. Run on all edges then filter result for the edges you want.</h4>
</div>

In [72]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import plotly.express as px
from shapely import wkt
import geopandas as gpd
from pathlib import Path

<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ do not modify ★ </h3>
  <p> This notebook is used to consolidate mappings back into edges and give each edge a score. </p>
</div>

In [73]:
def cb_scoring(source,
                mapping_csv,
                blend_light=False,
                decay_lambda=0.01,
                light_alpha = 0.2,
                cross_beta = 0.4,
                time_col='Offense DateTime',
                is_day_col = "is_day",
                crime_score_col = 'Crime Score',
                max_score=10.0,
                segment_id_col='edge_id',
                segment_length_col='length'
                ):
    """
        source: projected edges_crs file
        mapping_csv: csv file with the mappings of crime points to edges
        blend_light: inclusion of light data
        decay_lambda: coefficient of decay for recency calculations
        light_alpha: weight of light on score
        cross_beta: weight of opposite time of day on score

        csv columns:
            time_col: column of csv that has the datetime of the crime
            is_day_col: column of csv that indicates crime occurred in day or night
            crime_score_col: column of csv that contains the crime severity
            max_score: max score of the crime score
            segment_id_col: column of csv that is the index to join segments on
            segment_length_col: column of csv that indicateslength of the segment

        scoring method:
        for each segment (osm_id), calculate:
        raw_risk = sum(recency * severity) * (count / min(length,150))
        capped_risk = min(1, raw_risk)
        safety_score = 1 - capped_risk

    """

    # read edges from csv and prepare index
    edges = source

    if segment_id_col not in edges.columns:
        edges = edges.reset_index().rename(columns={'index': segment_id_col})
    
    # get crime mappings from csv
    crime_df = mapping_csv

    # calculate recency and severity
    now = pd.Timestamp.now()
    crime_df['delta_days'] = (now - pd.to_datetime(crime_df[time_col])).dt.days
    crime_df['recency'] = np.exp(-decay_lambda * crime_df['delta_days'])
    crime_df['crime_score_raw'] = pd.to_numeric(crime_df[crime_score_col], errors='coerce').fillna(0.0)
    crime_df['crime_weight'] = (crime_df['crime_score_raw'] / float(max_score)).clip(0, 1)

    def _coerce_bool_series(s):
        # Works for bools, ints, and strings like "true"/"False"/"0"/"1"
        return s.map(lambda v: (
            bool(v) if isinstance(v, (bool, np.bool_))
            else str(v).strip().lower() in {"1","true","t","yes","y"}
        ))
    # identify if day crime or night crime
    crime_df[is_day_col] = _coerce_bool_series(crime_df[is_day_col])

    # calculate the score according to the formula
    def score(subdf, length):
        count = len(subdf)
        if length <= 0 or count == 0:
            return 1.0
        weighted_sum = (subdf['recency'] * subdf['crime_weight']).sum()
        density = count / min(length, 150)
        raw_risk = weighted_sum * density
        capped = min(1.0, raw_risk)
        safety = 1.0 - capped
        return safety

    # for each edge, consolidate the day, night, and light densities
    results = []
    for _, edge in edges.iterrows():
        eid = edge[segment_id_col]
        length = edge[segment_length_col]

        seg_crimes = crime_df[crime_df[segment_id_col] == eid]

        seg_day = seg_crimes[seg_crimes[is_day_col]]
        seg_night = seg_crimes[~seg_crimes[is_day_col]]

        day_score = score(seg_day, length)
        night_score = score(seg_night, length)

        # add metadata of crimes for each segment
        def summarize(crimes_df):
            crime_type_col = 'Offense Type'
            df = crimes_df.copy()

            # ensure your timestamp column is datetime and df isnt empty
            if df.empty:
                return {}
            df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
            
            # gather counts of each crime type and the last time each crime type occurred
            counts = df[crime_type_col].value_counts().to_dict()
            last_ts = df.groupby(crime_type_col)[time_col].max()
            last = last_ts.dt.strftime('%Y-%m-%d %H:%M:%S').to_dict()
            
            # return summary
            return {
                crim_type: {
                    'count': counts[crim_type],
                    'last_at': last.get(crim_type)
                }
                for crim_type in counts
            }
        
        # append summaries for day and night as columns
        day_summary   = summarize(seg_day)
        night_summary = summarize(seg_night)

        results.append({
            segment_id_col: eid,
            segment_length_col: length,
            'day_score':  day_score,
            'night_score':night_score,
            'day_notes':   day_summary,
            'night_notes': night_summary
        })

    # build final df and merge back any other edge attributes
    scores_df = pd.DataFrame(results)

    # if true, calculate light score and apply to night score
    if blend_light:
        light = pd.read_csv("../Lightpoles/data/newlightpoles.csv")
        scores_df['ldensity'] = light['ldensity']
        light_max = scores_df['ldensity'].max()
        if not np.isfinite(light_max) or light_max <= 0:
            light_max = 1.0
        scores_df['light_score'] = (scores_df['ldensity'] / light_max).clip(0, 1)
        scores_df['night_score'] = (
            scores_df['night_score'] * (1 + light_alpha * scores_df['light_score'])
        ).clip(0, 1)

    # cross-period blending between day and night scores
    scores_df['blended_day_score'] = (
        scores_df['day_score'] - cross_beta * (1 - scores_df['night_score'])
    ).clip(0, 1)

    scores_df['blended_night_score'] = (
        scores_df['night_score'] - cross_beta * (1 - scores_df['day_score'])
    ).clip(0, 1)

    # merge into final df
    final = scores_df.merge(
        edges,
        on=segment_id_col,
        how='left',
        suffixes=('_calc','')
    )

    # tack columns onto the originl edges df
    edges['buffer_day'] = final['day_score'] * 0.9 + 0.1
    edges['buffer_night'] = final['night_score'] * 0.9 + 0.1
    edges['buffer_blended_day'] = final['blended_day_score'] * 0.9 + 0.1
    edges['buffer_blended_night'] = final['blended_night_score'] * 0.9 + 0.1
    edges['day_notes'] = final['day_notes'] 
    edges['night_notes'] = final['night_notes']

    # filter edges for footways only
    edges_filtered = edges[edges['highway'] == 'footway'].copy()
    return edges_filtered




<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ multiple scoring ★ </h3>
  <p> score multiple buffer sizes at once (requires mappings) </p>
</div>

In [74]:
def cb_scoring_multiple():
    #get scores for all 3 methods and with varying buffer sizes
    # stubs = (
    #     ["direct_mapping"]
    #     + [f"street_buffer_{d}m" for d in (10, 15, 20, 25)]
    #     + [f"point_buffer_{d}m"  for d in (10, 15, 20, 25)]
    # )

    # edges = pd.read_csv("data/edges/edges_seattle.csv")

    # results = {
    #     stub: cb_scoring(edges, mapping_csv=f"data/mappings/edges_with_safety_score_{stub}.csv", buf=d)
    #     .to_csv(f"data/scores/edges_with_safety_score_{stub}.csv", index=False)
    #     for stub in stubs
    # }

    sizes = (10, 15, 20, 25, 40, 50)
    edges = pd.read_csv("data/edges/edges_seattle_filtered.csv")

    # results = {}
    # for d in sizes:
    #     stub = f"point_buffer_{d}m"
    #     mapping_df = pd.read_csv(f"data/mappings/edges_with_safety_score_{stub}.csv")
    #     scored = cb_scoring(edges, mapping_csv=mapping_df)
    #     scored.to_csv(f"data/scores/edges_with_safety_score_{stub}.csv", index=False)
    #     results[stub] = scored


    # sizes = (10, 15, 20, 25)

    # results = {}
    # for d in sizes:
    #     stub = f"street_buffer_{d}m"
    #     mapping_df = pd.read_csv(f"data/mappings/edges_with_safety_score_{stub}.csv")
    #     scored = cb_scoring(edges, mapping_csv=mapping_df)
    #     scored.to_csv(f"data/scores/edges_with_safety_score_{stub}.csv", index=False)
    #     results[stub] = scored

    cb_scoring(edges, pd.read_csv("data/mappings/edges_with_safety_score_direct_mapping.csv")).to_csv("data/scores/edges_with_safety_score_direct_mapping.csv")

    # d = 10
    # stub = f"point_buffer_{d}m"
    # mapping_df = pd.read_csv(f"data/mappings/edges_with_safety_score_{stub}.csv")
    # scored = cb_scoring(edges, mapping_csv=mapping_df)
    # scored.to_csv(f"data/scores/edges_with_safety_score_{stub}.csv", index=False)


In [75]:
# edges = pd.read_csv("data/edges/edges_seattle_filtered.csv")
# cb_scoring(edges, pd.read_csv("data/mappings/edges_with_safety_score_direct_mapping.csv"))

In [76]:
cb_scoring_multiple()

<div style="
    text-align: center;
    max-width: 600px;
    margin: 0 auto;
    background:
    rgba(62, 128, 234, 1);
    color: white;
    padding: 12px;
    border-radius: 4px;
">
  <h3> ★ update ★ </h3>
  <p> simple call to updated csvs </p>
</div>

## Function call to quickly get updated data for the various buffer sizes and types

In [77]:
# %run edge_mapping.ipynb

# cb_mapping_multiple()
# cb_scoring_multiple()